# Major Project: Seasonal Agriculture Performance Analysis

**VOIS AICTE Batch 1 2026–2027**

### Project Objective
Analyze agricultural data from different seasons and identify meaningful patterns, trends, relationships, and differences in agricultural performance.

### Main Questions
1. How does agricultural performance vary across seasons?
2. What seasonal patterns can be observed in yield, production, revenue and profit?
3. How do environmental conditions differ across seasons?
4. How does resource usage vary across seasons?
5. What relationships exist between environmental/resource variables and agricultural outcomes?
6. Are there unusual or significant seasonal differences?
7. What recommendations can be made from the evidence?

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")

## 2. Load the Dataset

In [ ]:
# Upload the CSV to your notebook/Colab, then update the path if needed.
FILE_PATH = "seasonal_agriculture_performance_dataset (3).csv"

df = pd.read_csv(FILE_PATH)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Understand the Dataset

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.info())

In [ ]:
display(df.describe(include="all").T)

In [ ]:
print("Unique values in important categorical columns:")
for col in ["State", "District", "Crop", "Season", "Irrigation_Method"]:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head(10))

## 4. Data Quality Check

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

print("Missing values:")
display(missing.to_frame("Missing_Count"))

print("\nDuplicate rows:", df.duplicated().sum())

### Missing-value treatment
The supplied dataset contains missing values in **Rainfall_mm, Soil_Moisture_pct, and Yield_Tonnes_Ha**. For this analysis, numeric missing values are filled using the **median within the corresponding season** when possible. This reduces the influence of extreme values while preserving seasonal differences.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

for col in ["Rainfall_mm", "Soil_Moisture_pct", "Yield_Tonnes_Ha"]:
    df[col] = df.groupby("Season")[col].transform(
        lambda s: s.fillna(s.median())
    )

# Fallback in case any value is still missing
for col in ["Rainfall_mm", "Soil_Moisture_pct", "Yield_Tonnes_Ha"]:
    df[col] = df[col].fillna(df[col].median())

print("Remaining missing values:", df.isna().sum().sum())

## 5. Check Data Types and Valid Ranges

In [ ]:
print(df.dtypes)

range_checks = {
    "Farm_Area_Hectares": df["Farm_Area_Hectares"].min(),
    "Rainfall_mm": df["Rainfall_mm"].min(),
    "Avg_Temperature_C": df["Avg_Temperature_C"].min(),
    "Humidity_pct": df["Humidity_pct"].min(),
    "Soil_pH": df["Soil_pH"].min(),
    "Yield_Tonnes_Ha": df["Yield_Tonnes_Ha"].min(),
    "Profit_INR": df["Profit_INR"].min(),
}

display(pd.Series(range_checks, name="Minimum_Value"))

## 6. Descriptive Statistics by Season

In [ ]:
season_summary = df.groupby("Season").agg(
    Farms=("Farm_ID", "count"),
    Avg_Yield=("Yield_Tonnes_Ha", "mean"),
    Avg_Production=("Production_Tonnes", "mean"),
    Avg_Revenue=("Revenue_INR", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Water_Used=("Water_Used_m3", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Avg_Rainfall=("Rainfall_mm", "mean"),
    Avg_Temperature=("Avg_Temperature_C", "mean"),
    Avg_Humidity=("Humidity_pct", "mean"),
    Avg_Fertilizer=("Fertilizer_kg_ha", "mean"),
    Avg_Pesticide=("Pesticide_Litre_ha", "mean"),
    Avg_Risk=("Disease_Pest_Risk_pct", "mean")
).round(2)

display(season_summary)

## 7. Season-wise Yield Analysis

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Season", y="Yield_Tonnes_Ha")
plt.title("Yield Distribution Across Seasons")
plt.xlabel("Season")
plt.ylabel("Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

season_yield = df.groupby("Season")["Yield_Tonnes_Ha"].mean().sort_values(ascending=False)
display(season_yield.to_frame("Average_Yield_Tonnes_Ha"))

## 8. Production, Revenue and Profit Across Seasons

In [ ]:
figures = [
    ("Production_Tonnes", "Average Production by Season", "Production (Tonnes)"),
    ("Revenue_INR", "Average Revenue by Season", "Revenue (INR)"),
    ("Profit_INR", "Average Profit by Season", "Profit (INR)")
]

for col, title, ylabel in figures:
    plt.figure(figsize=(8,5))
    sns.barplot(data=df, x="Season", y=col, estimator="mean", errorbar=None)
    plt.title(title)
    plt.xlabel("Season")
    plt.ylabel(ylabel)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 9. Environmental Conditions by Season

In [ ]:
environmental_cols = [
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Humidity_pct",
    "Sunlight_Hours_Day",
    "Soil_pH",
    "Soil_Moisture_pct"
]

env_summary = df.groupby("Season")[environmental_cols].mean().round(2)
display(env_summary)

for col in environmental_cols:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x="Season", y=col)
    plt.title(f"{col} Across Seasons")
    plt.tight_layout()
    plt.show()

## 10. Resource Usage Across Seasons

In [ ]:
resource_cols = [
    "Nitrogen_kg_ha",
    "Phosphorus_kg_ha",
    "Potassium_kg_ha",
    "Fertilizer_kg_ha",
    "Pesticide_Litre_ha",
    "Water_Used_m3",
    "Water_Efficiency_t_per_1000m3"
]

resource_summary = df.groupby("Season")[resource_cols].mean().round(2)
display(resource_summary)

In [ ]:
for col in resource_cols:
    plt.figure(figsize=(8,5))
    sns.barplot(data=df, x="Season", y=col, estimator="mean", errorbar=None)
    plt.title(f"Average {col} by Season")
    plt.xlabel("Season")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

## 11. Crop-wise Seasonal Comparison

In [ ]:
crop_season_yield = pd.pivot_table(
    df,
    values="Yield_Tonnes_Ha",
    index="Crop",
    columns="Season",
    aggfunc="mean"
).round(2)

display(crop_season_yield)

plt.figure(figsize=(12,7))
sns.heatmap(crop_season_yield, annot=True, fmt=".2f")
plt.title("Average Crop Yield by Season")
plt.tight_layout()
plt.show()

## 12. Irrigation Method and Seasonal Performance

In [ ]:
irrigation_summary = df.groupby(["Season", "Irrigation_Method"]).agg(
    Average_Yield=("Yield_Tonnes_Ha", "mean"),
    Average_Water_Used=("Water_Used_m3", "mean"),
    Average_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Average_Profit=("Profit_INR", "mean")
).round(2)

display(irrigation_summary)

## 13. State-wise Seasonal Analysis

In [ ]:
state_season = df.groupby(["State", "Season"]).agg(
    Average_Yield=("Yield_Tonnes_Ha", "mean"),
    Average_Profit=("Profit_INR", "mean"),
    Average_Rainfall=("Rainfall_mm", "mean")
).round(2)

display(state_season.head(30))

## 14. Correlation Analysis

In [ ]:
corr_cols = [
    "Farm_Area_Hectares",
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Humidity_pct",
    "Sunlight_Hours_Day",
    "Soil_pH",
    "Soil_Moisture_pct",
    "Nitrogen_kg_ha",
    "Phosphorus_kg_ha",
    "Potassium_kg_ha",
    "Fertilizer_kg_ha",
    "Pesticide_Litre_ha",
    "Seed_Quality_Score",
    "Yield_Tonnes_Ha",
    "Production_Tonnes",
    "Market_Price_INR_Tonne",
    "Total_Cost_INR",
    "Revenue_INR",
    "Profit_INR",
    "Water_Used_m3",
    "Water_Efficiency_t_per_1000m3",
    "Disease_Pest_Risk_pct"
]

corr = df[corr_cols].corr()

plt.figure(figsize=(15,11))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

display(
    corr["Yield_Tonnes_Ha"]
    .drop("Yield_Tonnes_Ha")
    .sort_values(key=abs, ascending=False)
    .to_frame("Correlation_with_Yield")
)

## 15. Relationship Between Key Factors and Yield

In [ ]:
relationship_cols = [
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Soil_Moisture_pct",
    "Fertilizer_kg_ha",
    "Seed_Quality_Score",
    "Water_Used_m3",
    "Disease_Pest_Risk_pct"
]

for col in relationship_cols:
    plt.figure(figsize=(7,5))
    sns.scatterplot(data=df, x=col, y="Yield_Tonnes_Ha", hue="Season")
    plt.title(f"{col} vs Yield")
    plt.tight_layout()
    plt.show()

## 16. Statistical Test: Does Yield Differ by Season?

In [ ]:
groups = [
    group["Yield_Tonnes_Ha"].dropna().values
    for _, group in df.groupby("Season")
]

# One-way ANOVA
anova_result = stats.f_oneway(*groups)

print("One-way ANOVA")
print("F-statistic:", round(anova_result.statistic, 4))
print("p-value:", round(anova_result.pvalue, 6))

if anova_result.pvalue < 0.05:
    print("Result: Yield differences across seasons are statistically significant at the 5% level.")
else:
    print("Result: No statistically significant yield difference was detected across seasons at the 5% level.")

## 17. Statistical Test: Profit Differences by Season

In [ ]:
profit_groups = [
    group["Profit_INR"].dropna().values
    for _, group in df.groupby("Season")
]

profit_anova = stats.f_oneway(*profit_groups)

print("One-way ANOVA for Profit")
print("F-statistic:", round(profit_anova.statistic, 4))
print("p-value:", round(profit_anova.pvalue, 6))

if profit_anova.pvalue < 0.05:
    print("Result: Profit differs significantly across seasons at the 5% level.")
else:
    print("Result: No statistically significant profit difference was detected across seasons at the 5% level.")

## 18. Identify Highest and Lowest Seasonal Performance

In [ ]:
metrics = {
    "Yield": "Yield_Tonnes_Ha",
    "Production": "Production_Tonnes",
    "Revenue": "Revenue_INR",
    "Profit": "Profit_INR",
    "Water Efficiency": "Water_Efficiency_t_per_1000m3"
}

for name, col in metrics.items():
    means = df.groupby("Season")[col].mean()
    print(f"{name}:")
    print("  Highest:", means.idxmax(), "=", round(means.max(), 2))
    print("  Lowest :", means.idxmin(), "=", round(means.min(), 2))

## 19. Detect Unusual Seasonal Patterns

In [ ]:
# Compare each season's average yield with the overall average.
overall_yield = df["Yield_Tonnes_Ha"].mean()
season_yield = df.groupby("Season")["Yield_Tonnes_Ha"].mean()

unusual = ((season_yield - overall_yield) / overall_yield * 100).round(2)

display(
    unusual.to_frame("Yield_Difference_from_Overall_Percent")
)

# Highest disease/pest risk seasons
risk = df.groupby("Season")["Disease_Pest_Risk_pct"].mean().sort_values(ascending=False)
display(risk.to_frame("Average_Disease_Pest_Risk_pct"))

## 20. Automated Findings Summary

In [ ]:
season_means = df.groupby("Season").agg(
    Yield=("Yield_Tonnes_Ha", "mean"),
    Production=("Production_Tonnes", "mean"),
    Revenue=("Revenue_INR", "mean"),
    Profit=("Profit_INR", "mean"),
    Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Rainfall=("Rainfall_mm", "mean"),
    Soil_Moisture=("Soil_Moisture_pct", "mean"),
    Disease_Risk=("Disease_Pest_Risk_pct", "mean")
)

print("KEY FINDINGS")
print("=" * 60)

for metric in season_means.columns:
    highest = season_means[metric].idxmax()
    lowest = season_means[metric].idxmin()
    print(f"{metric}: highest in {highest}; lowest in {lowest}")

## 21. Conclusions

Use the numerical results and graphs above to write conclusions in this format:

- The analysis shows that agricultural performance changes across seasons.
- The season with the highest average yield was identified from the dataset.
- Differences in production, revenue and profit were compared across seasons.
- Environmental factors such as rainfall, temperature, humidity and soil moisture were examined.
- Resource usage, including fertilizer, pesticide and water usage, was compared.
- Correlation analysis was used to identify relationships between important variables and yield.
- Statistical testing was used to determine whether observed seasonal differences were statistically significant.
- Any unusual patterns should be reported only when supported by the analysis.

**Important:** Replace these general statements with the actual values produced by the notebook before submitting the final report.

## 22. Recommendations

Recommendations should be based on the actual findings. Possible recommendation categories include:

1. Improve seasonal crop planning based on observed yield patterns.
2. Use water more efficiently in seasons where water use is high but water efficiency is low.
3. Review fertilizer and pesticide usage where high resource usage does not correspond to better performance.
4. Pay attention to environmental conditions associated with lower agricultural performance.
5. Investigate regions or crops showing unusually low yield or profit.
6. Consider disease/pest risk while planning seasonal agricultural activities.
7. Use evidence from the dataset to support future resource allocation and seasonal planning.

## 23. Final Project Summary

This project analyzes seasonal agricultural performance using data related to crops, regions, environmental conditions, farming practices, resource usage and economic outcomes. The analysis includes data cleaning, exploratory analysis, visualization, correlation analysis and statistical testing. The final findings and recommendations should be based on the results generated from the supplied dataset.

### Submission Checklist
- [ ] Dataset loaded correctly
- [ ] Missing values handled
- [ ] Duplicate records checked
- [ ] Descriptive statistics completed
- [ ] Seasonal comparisons completed
- [ ] Environmental analysis completed
- [ ] Resource analysis completed
- [ ] Crop/state/irrigation comparisons completed
- [ ] Correlation analysis completed
- [ ] Statistical tests completed
- [ ] Findings written using actual results
- [ ] Recommendations connected to findings
- [ ] Notebook executed from beginning to end